# Swipe Atlas Final Workflow

Main project story: predict dating-app engagement (`mutual_matches`) and segment users by behavior. `match_outcome` is retained as a no-signal case study because the synthetic labels are balanced and not practically predictable from the available features.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
print(ROOT)

## 1. EDA and Signal Check

In [ ]:
subprocess.run([sys.executable, "scripts/01_eda.py"], check=True)
subprocess.run([sys.executable, "scripts/02_signal_test.py"], check=True)

## 2. Engagement Modeling

The official model predicts `mutual_matches` without using `likes_received` or `match_outcome`. A paired-feature comparison is generated separately to show leakage inflation.

In [ ]:
subprocess.run([sys.executable, "scripts/04_train_engagement_models.py"], check=True)

## 3. User Segmentation

In [ ]:
subprocess.run([sys.executable, "scripts/05_segmentation.py"], check=True)

## 4. AutoML Comparison Under Platform Constraints

> **auto-sklearn is Linux/Colab-only.** It cannot run locally on Windows because it depends on Python's Unix-specific `resource` module.  
> AutoGluon Tabular is the executable Windows AutoML comparison used for this project.
>
> **Why it matters:** the rubric (Step 7) asks how the selected model compares with auto-sklearn.  
> In this Windows-based workflow, report the real AutoGluon result and list auto-sklearn as not run locally unless you complete the optional Colab/Linux run.

### Option A - auto-sklearn 2.0 (optional Colab/Linux strict-rubric run)
Reference: Feurer, Eggensperger, Falkner, Lindauer & Hutter, JMLR 23(261):1-61, 2022

### Option B - AutoGluon 1.x (executed Windows-compatible AutoML comparison)
Reference: Gijsbers et al., JMLR 25(101):1-65, 2024 (OpenML AutoML Benchmark)

In [ ]:
# AutoML comparison under platform constraints
# This cell reports the executed Windows-compatible AutoGluon comparison.
# auto-sklearn remains an optional Colab/Linux-only backend.

from pathlib import Path
import pandas as pd

root = Path.cwd()
if not (root / "reports" / "automl_results.csv").exists() and (root.parent / "reports" / "automl_results.csv").exists():
    root = root.parent

results_path = root / "reports" / "automl_results.csv"
leaderboard_path = root / "reports" / "automl_leaderboard_autogluon.csv"

if not results_path.exists():
    raise FileNotFoundError(
        "Run `python scripts/06_automl_comparison.py --backend autogluon --sample 12000 --autogluon-time 600` first."
    )

results = pd.read_csv(results_path)
display(results)

if leaderboard_path.exists():
    leaderboard = pd.read_csv(leaderboard_path)
    display(leaderboard[["model", "score_test", "score_val", "eval_metric"]].head(10))

print("Report interpretation:")
print("AutoGluon best_quality achieved R2 approx 0.000 on the safe feature set, matching the dummy baseline and tuned manual model.")
print("Do not claim auto-sklearn confirmed the result unless it is later run in Colab/Linux.")


## 5. Report Tables

In [ ]:
import pandas as pd
display(pd.read_csv("reports/engagement_model_results.csv"))
display(pd.read_csv("reports/segmentation_summary.csv"))